# 03 — Baseline Models

_Placeholder — Standalone Random Forest and XGBoost baselines for comparison against the stacking ensemble._

In [1]:
%pip install xgboost imbalanced-learn shap

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, time
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score

PROCESSED = os.path.join(os.getcwd(), "..", "data", "processed", "preprocessed.joblib")
d = joblib.load(PROCESSED)
X_train, X_test = d["X_train"], d["X_test"]
y_train, y_test = d["y_train"], d["y_test"]
class_names = d["label_classes"]

print("X_train:", X_train.shape, " X_test:", X_test.shape)
print("Classes:", class_names)

def report(name, y_true, y_pred):
    print(f"\n===== {name} =====")
    print("Accuracy    :", round(accuracy_score(y_true, y_pred), 4))
    print("Macro-F1    :", round(f1_score(y_true, y_pred, average='macro'), 4))
    print("Weighted-F1 :", round(f1_score(y_true, y_pred, average='weighted'), 4))
    print(classification_report(y_true, y_pred, target_names=class_names,
                                digits=3, zero_division=0))

X_train: (2262300, 64)  X_test: (565576, 64)
Classes: ['BENIGN', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'Heartbleed', 'Infiltration', 'PortScan', 'SSH-Patator', 'Web Attack']


In [3]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=30,          # bounds tree size so RF fits comfortably in 16 GB
    min_samples_leaf=2,
    n_jobs=4,              # capped to limit per-process memory on Windows
    random_state=42,
)
t0 = time.time()
rf.fit(X_train, y_train)
print(f"RF trained in {time.time()-t0:.1f}s")

rf_pred = rf.predict(X_test)
report("Random Forest (no SMOTE)", y_test, rf_pred)

RF trained in 234.5s

===== Random Forest (no SMOTE) =====
Accuracy    : 0.9982
Macro-F1    : 0.943
Weighted-F1 : 0.9982
                  precision    recall  f1-score   support

          BENIGN      0.999     0.999     0.999    454265
             Bot      0.885     0.514     0.650       391
            DDoS      1.000     0.999     0.999     25605
   DoS GoldenEye      0.996     0.991     0.993      2059
        DoS Hulk      0.997     0.995     0.996     46025
DoS Slowhttptest      0.949     0.989     0.969      1100
   DoS slowloris      0.998     0.997     0.997      1159
     FTP-Patator      0.991     0.996     0.993      1587
      Heartbleed      1.000     1.000     1.000         2
    Infiltration      1.000     0.571     0.727         7
        PortScan      0.994     1.000     0.997     31761
     SSH-Patator      1.000     0.979     0.989      1179
      Web Attack      0.985     0.913     0.948       436

        accuracy                          0.998    565576
       

In [4]:
xgb = XGBClassifier(
    n_estimators=100,
    max_depth=8,
    learning_rate=0.3,
    tree_method="hist",
    device="cuda",         # uses your NVIDIA GPU
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
)
t0 = time.time()
xgb.fit(X_train, y_train)
print(f"XGBoost trained in {time.time()-t0:.1f}s")

xgb_pred = xgb.predict(X_test)
report("XGBoost (no SMOTE)", y_test, xgb_pred)

XGBoost trained in 27.7s


C:\Users\OMEN\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\xgboost\core.py:569: UserWarning: [10:24:53] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)



===== XGBoost (no SMOTE) =====
Accuracy    : 0.9989
Macro-F1    : 0.9487
Weighted-F1 : 0.9989
                  precision    recall  f1-score   support

          BENIGN      1.000     0.999     0.999    454265
             Bot      0.868     0.757     0.809       391
            DDoS      1.000     1.000     1.000     25605
   DoS GoldenEye      0.994     0.994     0.994      2059
        DoS Hulk      0.998     1.000     0.999     46025
DoS Slowhttptest      0.939     0.990     0.964      1100
   DoS slowloris      0.992     0.996     0.994      1159
     FTP-Patator      1.000     0.998     0.999      1587
      Heartbleed      1.000     1.000     1.000         2
    Infiltration      0.556     0.714     0.625         7
        PortScan      0.994     1.000     0.997     31761
     SSH-Patator      0.997     0.997     0.997      1179
      Web Attack      0.959     0.956     0.958       436

        accuracy                          0.999    565576
       macro avg      0.946     0